# Apache GitHub Analytics Pipeline

This notebook reads raw GitHub JSON data from a Databricks volume, cleans it with PySpark, and produces final analytics for the project submission.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

RAW_BASE_PATH = "/Volumes/workspace/github_project/github_raw/github"
ORG_NAME = "apache"
EXTRACT_DATE = "2026-05-15"
REPO_LIMIT = 200

def show_df(df, rows=20):
    if "display" in globals():
        display(df.limit(rows))
    else:
        df.show(rows, truncate=False)


## Bronze: Read Raw JSON

In [0]:
repos_raw = (
    spark.read.option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/org={ORG_NAME}/extract_date={EXTRACT_DATE}/source=repos/*.json")
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("extract_date", F.lit(EXTRACT_DATE))
)

commits_raw = (
    spark.read.option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/org={ORG_NAME}/extract_date={EXTRACT_DATE}/source=commits/repo=*/*.json")
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("extract_date", F.lit(EXTRACT_DATE))
    .withColumn("repo_name", F.regexp_extract(F.col("source_file"), r"repo=([^/]+)", 1))
)

contributors_raw = (
    spark.read.option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/org={ORG_NAME}/extract_date={EXTRACT_DATE}/source=contributors/repo=*/*.json")
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("extract_date", F.lit(EXTRACT_DATE))
    .withColumn("repo_name", F.regexp_extract(F.col("source_file"), r"repo=([^/]+)", 1))
)

print("Bronze loaded")
print("Repos:", repos_raw.count())
print("Commits:", commits_raw.count())
print("Contributors:", contributors_raw.count())


Bronze loaded
Repos: 200
Commits: 1537
Contributors: 12427


## Silver: Clean and Structure Data

In [0]:
repo_rank_window = Window.partitionBy("repo_id").orderBy(
    F.col("extract_date").desc(),
    F.col("pushed_at").desc()
)

repositories = (
    repos_raw.select(
        F.col("id").alias("repo_id"),
        F.col("name").alias("repo_name"),
        "full_name",
        "language",
        "stargazers_count",
        "forks_count",
        "open_issues_count",
        "archived",
        "disabled",
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("updated_at").alias("updated_at"),
        F.to_timestamp("pushed_at").alias("pushed_at"),
        "extract_date",
    )
    .withColumn("repo_rank", F.row_number().over(repo_rank_window))
    .filter(F.col("repo_rank") == 1)
    .filter(~F.col("archived") & ~F.col("disabled"))
    .withColumn(
        "activity_rank",
        F.row_number().over(
            Window.orderBy(F.col("pushed_at").desc(), F.col("repo_name").asc())
        ),
    )
    .filter(F.col("activity_rank") <= REPO_LIMIT)
    .drop("repo_rank", "activity_rank", "archived", "disabled")
)

commits = (
    commits_raw.select(
        "repo_name",
        "sha",
        F.col("author.login").alias("author_login"),
        F.col("author.id").alias("author_id"),
        F.col("commit.author.name").alias("commit_author_name"),
        F.col("commit.author.email").alias("commit_author_email"),
        F.to_timestamp(F.col("commit.author.date")).alias("commit_timestamp"),
        F.col("commit.message").alias("message"),
        "extract_date",
    )
    .join(repositories.select("repo_id", "repo_name"), on="repo_name", how="left")
    .dropDuplicates(["repo_id", "sha"])
)

contributors = (
    contributors_raw.select(
        "repo_name",
        F.col("login").alias("contributor_login"),
        F.col("id").alias("contributor_id"),
        F.col("name").alias("contributor_name"),
        "contributions",
        "type",
        F.col("extract_date").alias("snapshot_date"),
        F.when(F.col("login").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("is_anonymous"),
        F.coalesce(
            F.col("login"),
            F.concat_ws(
                ":",
                F.lit("anonymous"),
                F.coalesce(F.col("name"), F.lit("unknown")),
                F.col("contributions").cast("string"),
            ),
        ).alias("contributor_key"),
    )
    .join(repositories.select("repo_id", "repo_name"), on="repo_name", how="left")
    .dropDuplicates(["repo_id", "contributor_key", "snapshot_date", "is_anonymous"])
    .drop("contributor_key")
)

print("Silver created")
print("Repositories:", repositories.count())
print("Commits:", commits.count())
print("Contributors:", contributors.count())

show_df(repositories.select("repo_name", "language", "stargazers_count", "forks_count"))


Silver created


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Repositories: 200
Commits: 1537
Contributors: 12099


repo_name,language,stargazers_count,forks_count
fineract-backoffice-ui,TypeScript,3,5
datafusion-comet,Scala,1187,319
airavata-custos,C++,20,31
axis-site,HTML,5,2
axis-axis2-java-transports,Java,1,0
axis-axis2-java-savan,Java,1,0
axis-axis2-java-sandesha,Java,1,1
axis-axis2-java-rampart,Java,3,7
axis-axis2-java-kandula,Java,1,0
axis-axis2-java-core,Java,42,49


## Gold: Final Analytics

In [0]:
repo_activity = (
    commits.groupBy("repo_name")
    .agg(F.count("*").alias("commit_count_30d"))
    .orderBy(F.col("commit_count_30d").desc())
)

weekly_trends = (
    commits.withColumn("activity_week", F.date_trunc("week", F.col("commit_timestamp")))
    .groupBy("repo_name", "activity_week")
    .agg(F.count("*").alias("commit_count"))
    .orderBy("repo_name", "activity_week")
)

monthly_trends = (
    commits.withColumn("activity_month", F.date_trunc("month", F.col("commit_timestamp")))
    .groupBy("repo_name", "activity_month")
    .agg(F.count("*").alias("commit_count"))
    .orderBy("repo_name", "activity_month")
)

language_distribution = (
    repositories.groupBy("language")
    .agg(F.count("*").alias("repository_count"))
    .fillna({"language": "Unknown"})
    .orderBy(F.col("repository_count").desc())
)

contributor_counts = (
    contributors.groupBy("repo_name")
    .agg(F.count("*").alias("contributor_count_snapshot"))
    .orderBy(F.col("contributor_count_snapshot").desc())
)

popularity_vs_activity = (
    repositories.select("repo_name", "stargazers_count", "forks_count")
    .join(repo_activity, on="repo_name", how="left")
    .fillna({"commit_count_30d": 0})
    .orderBy(F.col("stargazers_count").desc())
)

first_seen_window = Window.partitionBy("repo_name", "author_login")
rank_window = Window.partitionBy("repo_name").orderBy(F.col("total_commits").desc(), F.col("author_login").asc())

contributor_timeline = (
    commits.filter(F.col("author_login").isNotNull())
    .withColumn("activity_week", F.date_trunc("week", F.col("commit_timestamp")))
    .groupBy("repo_name", "author_login", "activity_week")
    .agg(F.count("*").alias("weekly_commit_count"))
    .withColumn("first_active_week", F.min("activity_week").over(first_seen_window))
    .withColumn(
        "contributor_status",
        F.when(F.col("activity_week") == F.col("first_active_week"), F.lit("new")).otherwise(F.lit("returning")),
    )
)

contributor_concentration = (
    commits.filter(F.col("author_login").isNotNull())
    .groupBy("repo_name", "author_login")
    .agg(F.count("*").alias("total_commits"))
    .withColumn("contributor_rank", F.row_number().over(rank_window))
    .withColumn("repo_commit_total", F.sum("total_commits").over(Window.partitionBy("repo_name")))
    .withColumn("contribution_share", F.col("total_commits") / F.col("repo_commit_total"))
)

contributor_behavior = (
    contributor_timeline.join(
        contributor_concentration.select(
            "repo_name",
            "author_login",
            "contributor_rank",
            "total_commits",
            "repo_commit_total",
            "contribution_share",
        ),
        on=["repo_name", "author_login"],
        how="left",
    )
    .orderBy("repo_name", "activity_week", "contributor_rank")
)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


## Results

In [0]:
print("1. Most active repositories")
show_df(repo_activity)

print("2. Weekly trends")
show_df(weekly_trends)

print("3. Language distribution")
show_df(language_distribution)

print("4. Contributor counts")
show_df(contributor_counts)

print("5. Stars vs activity")
show_df(popularity_vs_activity)

print("6. Contributor behavior over time")
show_df(contributor_behavior)

1. Most active repositories


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


repo_name,commit_count_30d
airflow,776
spark,335
iceberg,180
kafka,140
flink,106


2. Weekly trends


repo_name,activity_week,commit_count
airflow,2026-04-13T00:00:00.000Z,73
airflow,2026-04-20T00:00:00.000Z,174
airflow,2026-04-27T00:00:00.000Z,202
airflow,2026-05-04T00:00:00.000Z,205
airflow,2026-05-11T00:00:00.000Z,122
flink,2026-04-13T00:00:00.000Z,18
flink,2026-04-20T00:00:00.000Z,30
flink,2026-04-27T00:00:00.000Z,22
flink,2026-05-04T00:00:00.000Z,19
flink,2026-05-11T00:00:00.000Z,17


...

repo_name,contributor_count_snapshot
airflow,4294
spark,3337
flink,2012
kafka,1645
iceberg,811


5. Stars vs activity


repo_name,stargazers_count,forks_count,commit_count_30d
superset,72853,17300,0
airflow,45428,17061,776
spark,43273,29182,335
kafka,32601,15192,140
pulsar,15241,3731,0
thrift,10921,4107,0
datafusion,8764,2091,0
beam,8587,4555,0
tomcat,8171,5376,0
camel,6205,5107,0


6. Contributor behavior over time


repo_name,author_login,activity_week,weekly_commit_count,first_active_week,contributor_status,contributor_rank,total_commits,repo_commit_total,contribution_share
airflow,potiuk,2026-04-13T00:00:00.000Z,21,2026-04-13T00:00:00.000Z,new,1,113,776,0.14561855670103094
airflow,dependabot[bot],2026-04-13T00:00:00.000Z,4,2026-04-13T00:00:00.000Z,new,2,42,776,0.05412371134020619
airflow,amoghrajesh,2026-04-13T00:00:00.000Z,2,2026-04-13T00:00:00.000Z,new,5,24,776,0.030927835051546393
airflow,jscheffl,2026-04-13T00:00:00.000Z,4,2026-04-13T00:00:00.000Z,new,6,23,776,0.029639175257731958
airflow,kaxil,2026-04-13T00:00:00.000Z,4,2026-04-13T00:00:00.000Z,new,7,23,776,0.029639175257731958
airflow,henry3260,2026-04-13T00:00:00.000Z,8,2026-04-13T00:00:00.000Z,new,8,21,776,0.027061855670103094
airflow,shahar1,2026-04-13T00:00:00.000Z,1,2026-04-13T00:00:00.000Z,new,9,19,776,0.024484536082474227
airflow,wjddn279,2026-04-13T00:00:00.000Z,2,2026-04-13T00:00:00.000Z,new,10,15,776,0.019329896907216496
airflow,ephraimbuddy,2026-04-13T00:00:00.000Z,2,2026-04-13T00:00:00.000Z,new,11,14,776,0.01804123711340206
airflow,jason810496,2026-04-13T00:00:00.000Z,2,2026-04-13T00:00:00.000Z,new,14,11,776,0.014175257731958763


## Final Checks

In [0]:
print("Duplicate commits:", commits.groupBy("repo_id", "sha").count().filter("count > 1").count())
print("Null commit timestamps:", commits.filter(F.col("commit_timestamp").isNull()).count())
print("Anonymous contributors:", contributors.filter(F.col("is_anonymous")).count())

summary_df = spark.createDataFrame(
    [
        ("repositories", repositories.count()),
        ("commits", commits.count()),
        ("contributors", contributors.count()),
        ("repo_activity", repo_activity.count()),
        ("weekly_trends", weekly_trends.count()),
        ("monthly_trends", monthly_trends.count()),
        ("language_distribution", language_distribution.count()),
        ("contributor_counts", contributor_counts.count()),
        ("popularity_vs_activity", popularity_vs_activity.count()),
        ("contributor_behavior", contributor_behavior.count()),
    ],
    ["dataset", "row_count"],
)

show_df(summary_df)

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicate commits: 0
Null commit timestamps: 0
Anonymous contributors: 10329


dataset,row_count
repositories,200
commits,1537
contributors,12099
repo_activity,5
weekly_trends,25
monthly_trends,10
language_distribution,25
contributor_counts,5
popularity_vs_activity,200
contributor_behavior,724
